In [2]:
import pandas as pd
import numpy as np

# 0. Cargar Datos

In [3]:
path_ventas = "datos/Ventas por Cliente/ventas_con_precio_lista_y_descuentos_2025_2026.csv"
columnas_ventas = [
    "Año",
    "Mes",
    "Cod Cliente",
    "Cod SKU",
    "Nombre Consolidado",
    "Kilo Real",
    "Monto Real",
]

ventas = pd.read_csv(
    path_ventas,
    usecols=columnas_ventas,
    dtype={"Cod Cliente": "string", "Cod SKU": "string"}
)

print("Ventas - Filas:", len(ventas))
ventas.head()

Ventas - Filas: 10596574


,Año,Mes,Cod Cliente,Nombre Consolidado,Cod SKU,Kilo Real,Monto Real
0,2025,1,1145633,COBERTURA,3034,4.0,43440
1,2025,1,43790,HORECA VOLUMEN,399,12.0,31608
2,2025,1,1227049,OTROS HORECA,1006,30.4,82749
3,2025,1,1174760,COBERTURA,3991,2.4,10862
4,2025,1,1015869,HORECA VOLUMEN,1446,4.8,15048


In [4]:
ventas.columns

Index(['Año', 'Mes', 'Cod Cliente', 'Nombre Consolidado', 'Cod SKU',
       'Kilo Real', 'Monto Real'],
      dtype='str')

In [5]:
canales_relevantes = [
    "COBERTURA",
    "VOLUMEN COBERTURA",
    "MAYORISTAS CADENAS",
    "MAYORISTA B VOLUMEN",
    "OTROS MAYORISTAS",
    "HORECA VOLUMEN",
    "OTROS HORECA",
]

ventas = ventas[ventas["Nombre Consolidado"].isin(canales_relevantes)].copy()
print("Ventas después de filtrar canales relevantes - Filas:", len(ventas))
ventas.head()

Ventas después de filtrar canales relevantes - Filas: 10364933


,Año,Mes,Cod Cliente,Nombre Consolidado,Cod SKU,Kilo Real,Monto Real
0,2025,1,1145633,COBERTURA,3034,4.0,43440
1,2025,1,43790,HORECA VOLUMEN,399,12.0,31608
2,2025,1,1227049,OTROS HORECA,1006,30.4,82749
3,2025,1,1174760,COBERTURA,3991,2.4,10862
4,2025,1,1015869,HORECA VOLUMEN,1446,4.8,15048


In [6]:
path_pedidos = "datos/Pedidos 2025 Julio - 2026 Marzo.csv"

pedidos = pd.read_csv(
    path_pedidos,
    sep=";",
    decimal=",",
    dtype={"CodCliente": "string", "CodSKU": "string"},
)

print("Pedidos - Filas:", len(pedidos))
pedidos.head()

Pedidos - Filas: 4335388


,Año,Mes,CodCliente,CodSKU,Valor,Kilos
0,2025,7,1000023,2621,9696.0,2.00
1,2025,7,1000023,3034,10472.0,1.00
2,2025,7,1000023,3038,16548.0,2.00
3,2025,7,1000023,3071,7757.0,1.60
4,2025,7,1000023,3239,199136.0,27.51


In [7]:
pedidos.columns

Index(['Año', 'Mes', 'CodCliente', 'CodSKU', 'Valor', 'Kilos'], dtype='str')

# 1. Preparar Llaves y Periodo de Cruce

In [8]:
def normalizar_codigo(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

ventas["Cod Cliente"] = normalizar_codigo(ventas["Cod Cliente"])
ventas["Cod SKU"] = normalizar_codigo(ventas["Cod SKU"])
pedidos["CodCliente"] = normalizar_codigo(pedidos["CodCliente"])
pedidos["CodSKU"] = normalizar_codigo(pedidos["CodSKU"])

meses_pedidos = pedidos[["Año", "Mes"]].drop_duplicates().sort_values(["Año", "Mes"])
ventas_periodo = ventas.merge(meses_pedidos, on=["Año", "Mes"], how="inner")

print("Meses presentes en pedidos:")
meses_pedidos.reset_index(drop=True)

Meses presentes en pedidos:


,Año,Mes
0,2025,7
1,2025,8
2,2025,9
3,2025,10
4,2025,11
5,2025,12
6,2026,1
7,2026,2
8,2026,3


# 2. Agregar Ventas y Pedidos

In [9]:
ventas_sku_cliente_mes = (
    ventas_periodo
    .groupby(["Año", "Mes", "Cod Cliente", "Cod SKU"], as_index=False)
    .agg(
        kilos_venta=("Kilo Real", "sum"),
        monto_venta=("Monto Real", "sum"),
    )
    .rename(columns={"Cod Cliente": "CodCliente", "Cod SKU": "CodSKU"})
)

pedidos_sku_cliente_mes = (
    pedidos
    .groupby(["Año", "Mes", "CodCliente", "CodSKU"], as_index=False)
    .agg(
        kilos_pedido=("Kilos", "sum"),
        monto_pedido=("Valor", "sum"),
    )
)

print("Filas agregadas ventas:", len(ventas_sku_cliente_mes))
print("Filas agregadas pedidos:", len(pedidos_sku_cliente_mes))
ventas_sku_cliente_mes.head()

Filas agregadas ventas: 3491526
Filas agregadas pedidos: 4335388


,Año,Mes,CodCliente,CodSKU,kilos_venta,monto_venta
0,2025,7,1000023,2621,2.000,9696
1,2025,7,1000023,3034,1.000,10472
2,2025,7,1000023,3038,2.000,16548
3,2025,7,1000023,3071,1.600,7757
4,2025,7,1000023,3239,27.505,199136


In [10]:
pedidos_sku_cliente_mes.head()

,Año,Mes,CodCliente,CodSKU,kilos_pedido,monto_pedido
0,2025,7,1000023,2621,2.00,9696.0
1,2025,7,1000023,3034,1.00,10472.0
2,2025,7,1000023,3038,2.00,16548.0
3,2025,7,1000023,3071,1.60,7757.0
4,2025,7,1000023,3239,27.51,199136.0


# 3. Comparar Ventas vs Pedidos

In [11]:
comparacion = (
    ventas_sku_cliente_mes
    .merge(
        pedidos_sku_cliente_mes,
        on=["Año", "Mes", "CodCliente", "CodSKU"],
        how="outer",
        indicator=True,
    )
    .rename(columns={"_merge": "origen"})
)

for col in ["kilos_venta", "monto_venta", "kilos_pedido", "monto_pedido"]:
    comparacion[col] = comparacion[col].fillna(0)

comparacion["dif_kilos"] = comparacion["kilos_venta"] - comparacion["kilos_pedido"]
comparacion["dif_monto"] = comparacion["monto_venta"] - comparacion["monto_pedido"]
comparacion["abs_dif_kilos"] = comparacion["dif_kilos"].abs()
comparacion["abs_dif_monto"] = comparacion["dif_monto"].abs()
comparacion["coincide_exacto"] = (
    comparacion["dif_kilos"].eq(0) & comparacion["dif_monto"].eq(0)
)
comparacion["coincide_con_redondeo_kilos"] = (
    np.isclose(comparacion["kilos_venta"], comparacion["kilos_pedido"], atol=0.01)
    & comparacion["dif_monto"].eq(0)
)

comparacion.head()

,Año,Mes,CodCliente,CodSKU,kilos_venta,monto_venta,kilos_pedido,monto_pedido,origen,dif_kilos,dif_monto,abs_dif_kilos,abs_dif_monto,coincide_exacto,coincide_con_redondeo_kilos
0,2025,7,1000023,2621,2.000,9696.0,2.00,9696.0,both,0.000,0.0,0.000,0.0,True,True
1,2025,7,1000023,3034,1.000,10472.0,1.00,10472.0,both,0.000,0.0,0.000,0.0,True,True
2,2025,7,1000023,3038,2.000,16548.0,2.00,16548.0,both,0.000,0.0,0.000,0.0,True,True
3,2025,7,1000023,3071,1.600,7757.0,1.60,7757.0,both,0.000,0.0,0.000,0.0,True,True
4,2025,7,1000023,3239,27.505,199136.0,27.51,199136.0,both,-0.005,0.0,0.005,0.0,False,True


In [16]:
faltan_kilos = comparacion[
    (comparacion["coincide_con_redondeo_kilos"] == False)
    & (comparacion["dif_kilos"] < 0)
]
faltan_kilos.head()

,Año,Mes,CodCliente,CodSKU,kilos_venta,monto_venta,kilos_pedido,monto_pedido,origen,dif_kilos,dif_monto,abs_dif_kilos,abs_dif_monto,coincide_exacto,coincide_con_redondeo_kilos
12,2025,7,1000032,3313,0.000,0.0,3.00,13961.0,right_only,-3.000,-13961.0,3.000,13961.0,False,False
14,2025,7,1000032,421,1.000,3877.0,2.00,7755.0,both,-1.000,-3878.0,1.000,3878.0,False,False
17,2025,7,1000032,8346,2.400,10085.0,4.80,20169.0,both,-2.400,-10084.0,2.400,10084.0,False,False
19,2025,7,1000032,8777,3.000,8145.0,6.00,16289.0,both,-3.000,-8144.0,3.000,8144.0,False,False
45,2025,7,1000093,1228,18.675,101402.0,18.69,101402.0,both,-0.015,0.0,0.015,0.0,False,False


In [12]:
resumen = pd.DataFrame(
    {
        "metrica": [
            "filas_comparadas",
            "solo_en_ventas",
            "solo_en_pedidos",
            "presentes_en_ambas",
            "diferencia_en_kilos",
            "diferencia_en_monto",
            "coinciden_exacto",
            "coinciden_con_redondeo_kilos",
        ],
        "valor": [
            len(comparacion),
            comparacion["origen"].eq("left_only").sum(),
            comparacion["origen"].eq("right_only").sum(),
            comparacion["origen"].eq("both").sum(),
            (~np.isclose(comparacion["dif_kilos"], 0, atol=1e-9)).sum(),
            comparacion["dif_monto"].ne(0).sum(),
            comparacion["coincide_exacto"].sum(),
            comparacion["coincide_con_redondeo_kilos"].sum(),
        ],
    }
)

resumen

,metrica,valor
0,filas_comparadas,4371252
1,solo_en_ventas,35864
2,solo_en_pedidos,879726
3,presentes_en_ambas,3455662
4,diferencia_en_kilos,1462154
5,diferencia_en_monto,1050995
6,coinciden_exacto,2839957
7,coinciden_con_redondeo_kilos,3291775


In [13]:
diferencias = comparacion[
    (comparacion["origen"] != "both")
    | (~np.isclose(comparacion["dif_kilos"], 0, atol=1e-9))
    | (comparacion["dif_monto"] != 0)
].copy()

diferencias = diferencias.sort_values(
    ["abs_dif_monto", "abs_dif_kilos"],
    ascending=[False, False],
)

print("Filas con diferencias o faltantes:", len(diferencias))
diferencias.head(20)

Filas con diferencias o faltantes: 1615357


,Año,Mes,CodCliente,CodSKU,kilos_venta,monto_venta,kilos_pedido,monto_pedido,origen,dif_kilos,dif_monto,abs_dif_kilos,abs_dif_monto,coincide_exacto,coincide_con_redondeo_kilos
2481980,2025,12,1039101,8454,0.0,0.0,126808.00,236243304.0,right_only,-126808.00,-236243304.0,126808.00,236243304.0,False,False
2481934,2025,12,1039101,1032,0.0,0.0,107408.00,220078992.0,right_only,-107408.00,-220078992.0,107408.00,220078992.0,False,False
1505920,2025,10,1039101,1032,0.0,0.0,104363.03,213833107.0,right_only,-104363.03,-213833107.0,104363.03,213833107.0,False,False
1026329,2025,9,1039101,8454,0.0,0.0,113824.00,212054112.0,right_only,-113824.00,-212054112.0,113824.00,212054112.0,False,False
1505970,2025,10,1039101,8454,0.0,0.0,104840.00,195309240.0,right_only,-104840.00,-195309240.0,104840.00,195309240.0,False,False
2980300,2026,1,1039101,8454,0.0,0.0,102888.00,191680344.0,right_only,-102888.00,-191680344.0,102888.00,191680344.0,False,False
3460881,2026,2,1039101,8454,0.0,0.0,102568.00,191084184.0,right_only,-102568.00,-191084184.0,102568.00,191084184.0,False,False
46645,2025,7,1039101,1032,0.0,0.0,98784.00,181715520.0,right_only,-98784.00,-181715520.0,98784.00,181715520.0,False,False
2980249,2026,1,1039101,1032,0.0,0.0,80752.00,181387808.0,right_only,-80752.00,-181387808.0,80752.00,181387808.0,False,False
542527,2025,8,1039101,8454,0.0,0.0,96696.00,180144648.0,right_only,-96696.00,-180144648.0,96696.00,180144648.0,False,False


# 4. Exportar Resultado

In [ ]:
path_salida_comparacion = "datos/comparacion_pedidos_vs_ventas_sku_cliente_mes.csv"
path_salida_diferencias = "datos/diferencias_pedidos_vs_ventas_sku_cliente_mes.csv"

comparacion.to_csv(path_salida_comparacion, index=False)
diferencias.to_csv(path_salida_diferencias, index=False)

print(f"Archivo guardado: {path_salida_comparacion}")
print(f"Archivo guardado: {path_salida_diferencias}")